# C6 迁移学习、检测框与 IoU

**竞赛模块：C · 深度学习技术应用** · **建议学习约 40 分钟**

CNN 分类之后，检测预测『框在哪』，分割预测『每个像素是哪类』。先掌握框坐标与 IoU 再碰大型网络。

> 零基础学习顺序：先读术语和说明 → 运行示例 → 理解代码 → 完成练习 → 概念自测。按 Shift+Enter 运行单元格。

## 一、先认识术语

实际运行 ResNet18 的冻结、替换分类头、单批次更新，再计算检测框 IoU。切记随机骨干不等于真正的预训练模型。
- **检测框 xyxy**：左上角 (x1,y1) 和右下角 (x2,y2) 的坐标
- **IoU**：交集面积 / 并集面积，是检测和分割的重要重合度指标

迁移学习常冻结已有骨干并替换分类头；目标检测返回框和类别；IoU = 两个框的交集面积 / 并集面积。语义分割输出每像素类别。

## 二、运行示例

逐格执行并查看输出。这里的代码使用本地/合成数据，遇到缺失输入会有提示。不要直接跳过基础代码。

In [ ]:
import torch
from torch import nn
from torchvision.models import resnet18

# 示范冻结与替换分类头：weights=None 是随机权重，并非真正预训练
torch.manual_seed(42)
model = resnet18(weights=None)
for param in model.parameters():
    param.requires_grad_(False)
model.fc = nn.Linear(model.fc.in_features, 2)
optimizer = torch.optim.SGD(model.fc.parameters(), lr=0.01)
model.eval()  # 保持随机初始化骨干的 BatchNorm 统计不变
inputs = torch.rand(2, 3, 64, 64)
labels = torch.tensor([0, 1])
optimizer.zero_grad()
scores = model(inputs)
loss = nn.CrossEntropyLoss()(scores, labels)
loss.backward()
optimizer.step()
print("可训练参数:", [n for n,p in model.named_parameters() if p.requires_grad])
print("分类输出形状:", tuple(scores.shape))

# 计算两个检测框交并比
box_a = torch.tensor([0., 0., 4., 4.])
box_b = torch.tensor([2., 2., 6., 6.])
inter_w = max(0., float(min(box_a[2],box_b[2])-max(box_a[0],box_b[0])))
inter_h = max(0., float(min(box_a[3],box_b[3])-max(box_a[1],box_b[1])))
inter = inter_w * inter_h
iou = inter / (16 + 16 - inter)
print("IoU:", iou)

## 三、逐句解释

1. weights=None 创建随机初始化网络结构；冻结全部参数后仍需替换分类头。

2. fc 替换为 2 类输出，优化器只更新 model.fc.parameters()。

3. model.eval() 保持 BatchNorm 统计固定；即使在 eval 状态也能对 fc 求梯度，本示例仅演示冻结机制。

4. 两个 4×4 框交集为 2×2，IoU=4/(16+16-4)=1/7。


## 四、重点函数与参数

### torchvision.models.resnet18

- **写法**：`models.resnet18(weights=None)`
- **参数**：weights=None 表示随机初始化，不是预训练！
- **返回**：返回 ResNet18 网络结构
- **特别注意**：迁移学习必须有真实加载的预训练权重才叫使用已学知识

### requires_grad

- **写法**：`parameter.requires_grad=False`
- **参数**：False：冻结该参数梯度
- **返回**：禁止训练该参数
- **特别注意**：替换分类头后重新选择可训练参数给优化器

## 五、运行结果核验

显示可训练参数只有 fc.weight、fc.bias；分类输出 (2,2)；IoU 约 0.142857。注意这不是具有实际迁移效果的预训练模型。

> 运行结果与模型随机性有关，请重点核验形状、指标和文件。


## 六、动手实践（先自己写）

**一级：入门**：手算例子 IoU=1/7。

**二级：巩固**：让 box_b 与 box_a 重合，观察 IoU=1。

**三级：挑战**：写一个计算两个 xyxy 框 IoU 的可重用函数。

**本课小挑战**：将 box_b 改为 box_a.clone()，重新计算 IoU=1；再观察哪些网络参数具有 requires_grad=True。

建议在下面的单元格中独立填写代码，先从一级入门开始，再做二级、三级。

In [ ]:
# TODO 1：完成一级入门练习

# 在这里输入代码


In [ ]:
# TODO 2：完成二级巩固练习

# 在这里输入代码


In [ ]:
# TODO 3：完成三级挑战练习

# 在这里输入代码


## 七、独立练习自动检查

让检测框完全重合，把 IoU 数值赋给 answer_c6。

独立编写 `answer_c6`，再运行检查单元。检查单元不包含答案；失败将显示提示，通过后保存本地结果。


In [ ]:
# TODO 4：完成 answer_c6 后运行
from pathlib import Path as _Path
import sys as _sys
for _candidate in [_Path.cwd(), *_Path.cwd().parents]:
    if (_candidate / "beginner_checkpoints.py").is_file():
        _sys.path.insert(0,str(_candidate))
        break
else:
    raise FileNotFoundError("未找到 beginner_checkpoints.py，请从项目根目录启动 Jupyter")
from beginner_checkpoints import check_lesson
check_lesson("C6",globals())


## 八、常见错误和排查

1. weights=None 只创建结构，不包含预训练知识。

2. 冻结参数后忘记重新创建 optimizer，可能导致优化器更新了错误参数。

**完成标准**：能够独立解释并运行本课完整示例。


## 九、参考写法（先完成练习再展开）

<details><summary>展开查看原练习参考代码</summary>

```python
box_b=box_a.clone()
print("重合框的 IoU 为 1")
```

参考写法不一定覆盖全部三级拓展练习；应以实际运行结果为准。

</details>

## 十、概念自测

**问题**：两个框完全重合时 IoU 为多少？

A. 0

B. 0.5

C. 1

<details><summary>展开答案与解析</summary>

正确选项：**C**。交集等于并集，所以 IoU=1。

</details>

## 十一、衔接竞赛实操

学完本模块基础课后，返回学习平台，进入 **6.3.4** 竞赛强化题。

[返回零基础课程网页](../../beginner_course.html#C6)

[打开 6.3.4 竞赛实操题](../../practice.html?question=6.3.4)